before starting genie code - select desired compute in top right, I use serverless v5 if allowed, otherwise use your own personal all purpose compute cluster

## objective of this notebook is to estimate the coverage of specific buildings in downtown seattle
- use tables created in notebook 01_Ingest and available in cmegdemos_catalog.network_enablement_demo schema
- Perform spatial joins with ST functions to determine which buildings fall within specific coverage hexagons
- calculate distance between buildings and cell towers
- do analysis on distance to a cell tower and signal strength of the h3 grid that the building falls in
- visualize with a heatmap of coverage for downtown seattle - make sure plot is scrollable/zoomable
- use databricks native ST functions and H3 indexing after converting data appropriately wherever possible
- ultimately we should save another table that contains all of this data for buildings in downtown seattle with the h3 coverage information and distance from nearest tower

prompt for genie code: see instructions in this notebook and get to work, feel free to add an additional markdown below with refined instructions after you create your plan and/or ask any questions to me before starting on the work

## Refined Analysis Plan

**Source tables** (from `01_ingest`):
| Table | Rows | Key Columns |
| --- | --- | --- |
| `fcc_bdc_5g_h3_seattle` | 22,737 | `h3_res9_id`, `mindown`, `minup`, `technology`, `environmnt` |
| `ms_building_footprints_seattle` | 584,310 | `geometry` (POLYGON 4326), `height` |
| `opencellid_tmobile_seattle` | 2,653 | `geometry` (POINT 4326), `radio`, `lat`, `lon` |

**Downtown Seattle bbox**: Lat 47.58–47.64, Lon -122.36 to -122.31 (~10.8K buildings)

### Analysis Steps
1. **Config** — parameterize catalog/schema/tables, downtown bbox
2. **Prepare buildings** — filter to downtown bbox, compute centroids, assign H3 res-9 index via `h3_longlatash3string()`
3. **H3 coverage join** — join building H3 index to FCC coverage on `h3_res9_id` → get `mindown`/`minup` per building
4. **Nearest tower** — cross join buildings×towers, `ST_DistanceSphere()`, window function → nearest tower distance + radio type per building
5. **Enriched table** — combine building + coverage + nearest tower into one DataFrame, summary statistics
6. **Heatmap** — folium interactive map (scrollable/zoomable) showing coverage quality and tower proximity
7. **Save** — persist enriched table to Delta for downstream use

### FCC Coverage Data Summary
| mindown | minup | environmnt | Hex count |
| --- | --- | --- | --- |
| 35 Mbps | 3 Mbps | 1 (outdoor) | 22,199 |
| 35 Mbps | 3 Mbps | 0 (indoor) | 422 |
| 7 Mbps | 1 Mbps | 0 (indoor) | 107 |
| 7 Mbps | 1 Mbps | 1 (outdoor) | 9 |

In [0]:
# ── Parameterized configuration ──────────────────────────────────────────────
CATALOG = "cmegdemos_catalog"
SCHEMA = "network_enablement_demo"

# Source tables (from 01_ingest)
FCC_TABLE = f"{CATALOG}.{SCHEMA}.fcc_bdc_5g_h3_seattle"
BUILDINGS_TABLE = f"{CATALOG}.{SCHEMA}.ms_building_footprints_seattle"
TOWERS_TABLE = f"{CATALOG}.{SCHEMA}.opencellid_tmobile_seattle"

# Output table
OUTPUT_TABLE = f"{CATALOG}.{SCHEMA}.downtown_seattle_building_coverage"

# Downtown Seattle bounding box
DT_LAT_MIN, DT_LAT_MAX = 47.58, 47.64
DT_LON_MIN, DT_LON_MAX = -122.36, -122.31

print(f"Source tables: {FCC_TABLE}, {BUILDINGS_TABLE}, {TOWERS_TABLE}")
print(f"Output table:  {OUTPUT_TABLE}")
print(f"Downtown bbox: lat [{DT_LAT_MIN}, {DT_LAT_MAX}], lon [{DT_LON_MIN}, {DT_LON_MAX}]")

### Step 1 — Prepare Downtown Buildings
Filter to downtown Seattle bbox, compute building centroids (lat/lon), assign H3 res-9 index for coverage join.

In [0]:
# Filter buildings to downtown Seattle and compute centroids + H3 indices
df_downtown = spark.sql(f"""
    SELECT
        monotonically_increasing_id() AS building_id,
        height,
        geometry AS building_geom,
        ST_Centroid(geometry) AS centroid,
        ST_X(ST_Centroid(geometry)) AS centroid_lon,
        ST_Y(ST_Centroid(geometry)) AS centroid_lat,
        h3_longlatash3string(
            ST_X(ST_Centroid(geometry)),
            ST_Y(ST_Centroid(geometry)),
            9
        ) AS h3_res9_id
    FROM {BUILDINGS_TABLE}
    WHERE ST_Y(ST_Centroid(geometry)) BETWEEN {DT_LAT_MIN} AND {DT_LAT_MAX}
      AND ST_X(ST_Centroid(geometry)) BETWEEN {DT_LON_MIN} AND {DT_LON_MAX}
""")

df_downtown.createOrReplaceTempView("downtown_buildings")
print(f"Downtown buildings: {df_downtown.count():,}")
display(df_downtown.limit(10))

### Step 2 — Join Buildings to FCC 5G NR Coverage
H3 index-based join: match each building’s H3 res-9 cell to FCC coverage data. Buildings outside any coverage hex will have NULL coverage fields.

In [0]:
# H3-based join: building h3_res9_id -> FCC coverage h3_res9_id
df_bldg_coverage = spark.sql(f"""
    SELECT
        b.*,
        f.technology,
        f.mindown AS min_download_mbps,
        f.minup AS min_upload_mbps,
        f.environmnt AS environment_type
    FROM downtown_buildings b
    LEFT JOIN {FCC_TABLE} f
        ON b.h3_res9_id = f.h3_res9_id
""")

df_bldg_coverage.createOrReplaceTempView("bldg_coverage")

total = df_bldg_coverage.count()
covered = df_bldg_coverage.filter("technology IS NOT NULL").count()
print(f"Total downtown buildings: {total:,}")
print(f"Buildings with 5G coverage: {covered:,} ({covered/total*100:.1f}%)")
print(f"Buildings without coverage:  {total - covered:,} ({(total-covered)/total*100:.1f}%)")

display(df_bldg_coverage.limit(10))

### Step 3 — Nearest T-Mobile Tower per Building
Cross join downtown buildings with towers, compute `ST_DistanceSphere()` (meters), then use `ROW_NUMBER()` window function to pick the nearest tower for each building.

In [0]:
# Find nearest tower for each downtown building using ST_DistanceSphere
df_nearest_tower = spark.sql(f"""
    WITH building_tower_distances AS (
        SELECT
            b.building_id,
            t.radio AS tower_radio,
            t.cell AS tower_cell_id,
            t.lat AS tower_lat,
            t.lon AS tower_lon,
            ST_DistanceSphere(b.centroid, t.geometry) AS distance_to_tower_m,
            ROW_NUMBER() OVER (
                PARTITION BY b.building_id
                ORDER BY ST_DistanceSphere(b.centroid, t.geometry)
            ) AS rn
        FROM downtown_buildings b
        CROSS JOIN {TOWERS_TABLE} t
    )
    SELECT
        building_id,
        tower_radio AS nearest_tower_radio,
        tower_cell_id AS nearest_tower_cell_id,
        tower_lat AS nearest_tower_lat,
        tower_lon AS nearest_tower_lon,
        distance_to_tower_m AS nearest_tower_distance_m
    FROM building_tower_distances
    WHERE rn = 1
""")

df_nearest_tower.createOrReplaceTempView("nearest_towers")
print(f"Nearest tower calculated for {df_nearest_tower.count():,} buildings")
display(df_nearest_tower.limit(10))

### Step 4 — Enriched Analysis Table
Combine building footprints + FCC H3 coverage + nearest tower distance into one comprehensive table. Add coverage quality classification.

In [0]:
# Combine coverage + nearest tower into enriched table
df_enriched = spark.sql("""
    SELECT
        bc.building_id,
        bc.height,
        bc.centroid_lat,
        bc.centroid_lon,
        bc.h3_res9_id,
        bc.building_geom,
        -- FCC coverage info
        bc.technology,
        bc.min_download_mbps,
        bc.min_upload_mbps,
        bc.environment_type,
        CASE
            WHEN bc.technology IS NULL THEN 'No Coverage'
            WHEN bc.min_download_mbps >= 35 THEN 'Strong (35+ Mbps)'
            WHEN bc.min_download_mbps >= 7 THEN 'Moderate (7+ Mbps)'
            ELSE 'Weak'
        END AS coverage_quality,
        -- Nearest tower info
        nt.nearest_tower_radio,
        nt.nearest_tower_cell_id,
        nt.nearest_tower_lat,
        nt.nearest_tower_lon,
        nt.nearest_tower_distance_m,
        CASE
            WHEN nt.nearest_tower_distance_m < 200 THEN 'Very Close (<200m)'
            WHEN nt.nearest_tower_distance_m < 500 THEN 'Close (200-500m)'
            WHEN nt.nearest_tower_distance_m < 1000 THEN 'Moderate (500m-1km)'
            ELSE 'Far (>1km)'
        END AS tower_proximity_band
    FROM bldg_coverage bc
    JOIN nearest_towers nt ON bc.building_id = nt.building_id
""")

df_enriched.createOrReplaceTempView("enriched_buildings")
print(f"Enriched buildings: {df_enriched.count():,}")

# Summary statistics
print("\n=== Coverage Quality Distribution ===")
display(spark.sql("""
    SELECT coverage_quality, COUNT(*) AS buildings, 
           ROUND(AVG(nearest_tower_distance_m), 0) AS avg_tower_dist_m
    FROM enriched_buildings
    GROUP BY coverage_quality
    ORDER BY buildings DESC
"""))

print("\n=== Tower Proximity Distribution ===")
display(spark.sql("""
    SELECT tower_proximity_band, COUNT(*) AS buildings,
           ROUND(AVG(min_download_mbps), 1) AS avg_download_mbps
    FROM enriched_buildings
    GROUP BY tower_proximity_band
    ORDER BY buildings DESC
"""))

print("\n=== Nearest Tower Radio Type ===")
display(spark.sql("""
    SELECT nearest_tower_radio, COUNT(*) AS buildings,
           ROUND(AVG(nearest_tower_distance_m), 0) AS avg_dist_m,
           ROUND(MIN(nearest_tower_distance_m), 0) AS min_dist_m,
           ROUND(MAX(nearest_tower_distance_m), 0) AS max_dist_m
    FROM enriched_buildings
    GROUP BY nearest_tower_radio
    ORDER BY buildings DESC
"""))

### Step 5 — Interactive Heatmap Visualization
Scrollable/zoomable folium map showing:
- **Heatmap layer**: building coverage quality (colored by download speed)
- **Tower markers**: T-Mobile tower locations with radio type labels
- **Building markers**: colored by coverage quality

In [0]:
%pip install folium -q

In [0]:
import folium
from folium.plugins import HeatMap
import pandas as pd

# Collect enriched building data to pandas
pdf = spark.sql("""
    SELECT centroid_lat, centroid_lon, height,
           COALESCE(min_download_mbps, 0) AS min_download_mbps,
           coverage_quality,
           nearest_tower_distance_m,
           tower_proximity_band
    FROM enriched_buildings
""").toPandas()

# Collect tower locations near downtown
pdf_towers = spark.sql(f"""
    SELECT lat, lon, radio, cell
    FROM {TOWERS_TABLE}
    WHERE lat BETWEEN {DT_LAT_MIN - 0.02} AND {DT_LAT_MAX + 0.02}
      AND lon BETWEEN {DT_LON_MIN - 0.02} AND {DT_LON_MAX + 0.02}
""").toPandas()

# Create base map centered on downtown Seattle
center_lat = (DT_LAT_MIN + DT_LAT_MAX) / 2
center_lon = (DT_LON_MIN + DT_LON_MAX) / 2
m = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles="cartodbpositron")

# --- Layer 1: Heatmap of coverage (weighted by download speed) ---
heat_data = [
    [row.centroid_lat, row.centroid_lon, row.min_download_mbps]
    for _, row in pdf.iterrows()
]
HeatMap(
    heat_data,
    name="Coverage Heatmap (download speed)",
    radius=12,
    blur=8,
    max_zoom=18,
    gradient={0.2: 'blue', 0.4: 'lime', 0.6: 'yellow', 0.8: 'orange', 1.0: 'red'}
).add_to(m)

# --- Layer 2: Building markers colored by coverage quality ---
coverage_colors = {
    'Strong (35+ Mbps)': 'green',
    'Moderate (7+ Mbps)': 'orange',
    'No Coverage': 'red'
}
bldg_group = folium.FeatureGroup(name="Buildings by Coverage", show=False)
for _, row in pdf.sample(min(2000, len(pdf))).iterrows():  # sample for performance
    color = coverage_colors.get(row.coverage_quality, 'gray')
    folium.CircleMarker(
        location=[row.centroid_lat, row.centroid_lon],
        radius=2,
        color=color,
        fill=True,
        fill_opacity=0.6,
        popup=f"DL: {row.min_download_mbps} Mbps<br>Tower dist: {row.nearest_tower_distance_m:.0f}m<br>Height: {row.height:.1f}m"
    ).add_to(bldg_group)
bldg_group.add_to(m)

# --- Layer 3: Tower markers ---
tower_colors = {'LTE': 'blue', 'NR': 'darkred', 'GSM': 'purple', 'UMTS': 'gray'}
tower_group = folium.FeatureGroup(name="T-Mobile Towers")
for _, row in pdf_towers.iterrows():
    color = tower_colors.get(row.radio, 'black')
    folium.Marker(
        location=[row.lat, row.lon],
        icon=folium.Icon(color=color, icon='signal', prefix='fa'),
        popup=f"{row.radio} Tower<br>Cell: {row.cell}"
    ).add_to(tower_group)
tower_group.add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Add legend
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
     background: white; padding: 10px; border-radius: 5px;
     border: 2px solid gray; font-size: 12px;">
  <b>Coverage Quality</b><br>
  <i style="color:green">●</i> Strong (35+ Mbps)<br>
  <i style="color:orange">●</i> Moderate (7+ Mbps)<br>
  <i style="color:red">●</i> No Coverage<br>
  <br><b>Tower Type</b><br>
  <i style="color:blue">●</i> LTE &nbsp;
  <i style="color:darkred">●</i> NR &nbsp;
  <i style="color:purple">●</i> GSM
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

print(f"Map: {len(pdf):,} buildings, {len(pdf_towers)} towers")
m

### Step 6 — Save Enriched Table to Delta
Persist the full enriched building coverage table for downstream analysis and dashboarding.

In [0]:
# Save enriched building coverage table (drop the geometry column for cleaner output)
df_to_save = spark.sql("""
    SELECT
        building_id,
        height,
        centroid_lat,
        centroid_lon,
        h3_res9_id,
        building_geom,
        technology,
        min_download_mbps,
        min_upload_mbps,
        environment_type,
        coverage_quality,
        nearest_tower_radio,
        nearest_tower_cell_id,
        nearest_tower_lat,
        nearest_tower_lon,
        nearest_tower_distance_m,
        tower_proximity_band
    FROM enriched_buildings
""")

df_to_save.write.mode("overwrite").saveAsTable(OUTPUT_TABLE)

print(f"✅ Saved {OUTPUT_TABLE}")
result = spark.sql(f"SELECT COUNT(*) AS rows FROM {OUTPUT_TABLE}").collect()[0]["rows"]
print(f"   Rows: {result:,}")

# Quick summary of saved table
print("\n=== Final Table Summary ===")
display(spark.sql(f"""
    SELECT
        coverage_quality,
        tower_proximity_band,
        COUNT(*) AS buildings,
        ROUND(AVG(nearest_tower_distance_m), 0) AS avg_tower_dist_m,
        ROUND(AVG(min_download_mbps), 1) AS avg_download_mbps,
        ROUND(AVG(height), 1) AS avg_height_m
    FROM {OUTPUT_TABLE}
    GROUP BY coverage_quality, tower_proximity_band
    ORDER BY coverage_quality, tower_proximity_band
"""))